In [1]:
import pandas as pd

In [9]:
colunas = ['NO_MUNICIPIO_PROVA', 'SG_UF_PROVA']
enem = pd.read_csv('microdados_enem_2023/DADOS/MICRODADOS_ENEM_2023.csv', usecols = colunas, sep = ';', encoding = 'ISO-8859-1')
enem.head()


,NO_MUNICIPIO_PROVA,SG_UF_PROVA
0,Brasília,DF
1,Brasília,DF
2,Caxias do Sul,RS
3,Fortaleza,CE
4,Quixadá,CE


In [10]:
enem = enem.drop_duplicates()
enem.head()

,NO_MUNICIPIO_PROVA,SG_UF_PROVA
0,Brasília,DF
2,Caxias do Sul,RS
3,Fortaleza,CE
4,Quixadá,CE
5,Ilhéus,BA


In [14]:
municipios = pd.read_csv('municipios.csv', sep = ';', encoding = 'UTF-8')
municipios.head()

,NO_MUNICIPIO_PROVA,SIGLA_UF
0,Abadia de Goiás,GO
1,Abadia dos Dourados,MG
2,Abadiânia,GO
3,Abaeté,MG
4,Abaetetuba,PA


In [15]:
# Realizando o merge entre os dois DataFrames
municipios['aplicou_prova'] = municipios['NO_MUNICIPIO_PROVA'].isin(enem['NO_MUNICIPIO_PROVA'])

# Convertendo os valores booleanos para "sim" ou "não"
municipios['aplicou_prova'] = municipios['aplicou_prova'].apply(lambda x: 'Sim' if x else 'Não')

# Exibindo o DataFrame atualizado
municipios.head()

,NO_MUNICIPIO_PROVA,SIGLA_UF,aplicou_prova
0,Abadia de Goiás,GO,Não
1,Abadia dos Dourados,MG,Não
2,Abadiânia,GO,Não
3,Abaeté,MG,Sim
4,Abaetetuba,PA,Sim


In [16]:
import geopandas as gpd

In [17]:
brasil = gpd.read_file('mapa_br.json')
print(brasil.head())

    CD_MUN                 NM_MUN SIGLA_UF  AREA_KM2  \
0  1100015  Alta Floresta D'Oeste       RO  7067.127   
1  1100023              Ariquemes       RO  4426.571   
2  1100031                 Cabixi       RO  1314.352   
3  1100049                 Cacoal       RO  3793.000   
4  1100056             Cerejeiras       RO  2783.300   

                                            geometry  
0  POLYGON ((-62.41771 -13.11894, -62.41052 -13.1...  
1  POLYGON ((-63.59697 -10.00038, -63.3555 -10.00...  
2  POLYGON ((-60.37162 -13.31859, -60.52408 -13.3...  
3  POLYGON ((-61.54898 -11.61911, -61.50059 -11.6...  
4  POLYGON ((-60.74296 -13.12235, -60.76267 -13.1...  


In [30]:
gdf_merged = brasil.merge(municipios[['NO_MUNICIPIO_PROVA', 'aplicou_prova', 'SIGLA_UF']], 
                          left_on=['NM_MUN', 'SIGLA_UF'], 
                          right_on=['NO_MUNICIPIO_PROVA', 'SIGLA_UF'], 
                          how='left')

# Salvar o GeoDataFrame atualizado como um novo GeoJSON
gdf_merged.head()


,CD_MUN,NM_MUN,SIGLA_UF,AREA_KM2,geometry,NO_MUNICIPIO_PROVA,aplicou_prova
0,1100015,Alta Floresta D'Oeste,RO,7067.127,"POLYGON ((-62.41771 -13.11894, -62.41052 -13.1...",Alta Floresta D'Oeste,Sim
1,1100023,Ariquemes,RO,4426.571,"POLYGON ((-63.59697 -10.00038, -63.3555 -10.00...",Ariquemes,Sim
2,1100031,Cabixi,RO,1314.352,"POLYGON ((-60.37162 -13.31859, -60.52408 -13.3...",Cabixi,Não
3,1100049,Cacoal,RO,3793.000,"POLYGON ((-61.54898 -11.61911, -61.50059 -11.6...",Cacoal,Sim
4,1100056,Cerejeiras,RO,2783.300,"POLYGON ((-60.74296 -13.12235, -60.76267 -13.1...",Cerejeiras,Sim


In [31]:
gdf_merged.drop(columns = 'NO_MUNICIPIO_PROVA', inplace = True)
gdf_merged.head()

,CD_MUN,NM_MUN,SIGLA_UF,AREA_KM2,geometry,aplicou_prova
0,1100015,Alta Floresta D'Oeste,RO,7067.127,"POLYGON ((-62.41771 -13.11894, -62.41052 -13.1...",Sim
1,1100023,Ariquemes,RO,4426.571,"POLYGON ((-63.59697 -10.00038, -63.3555 -10.00...",Sim
2,1100031,Cabixi,RO,1314.352,"POLYGON ((-60.37162 -13.31859, -60.52408 -13.3...",Não
3,1100049,Cacoal,RO,3793.000,"POLYGON ((-61.54898 -11.61911, -61.50059 -11.6...",Sim
4,1100056,Cerejeiras,RO,2783.300,"POLYGON ((-60.74296 -13.12235, -60.76267 -13.1...",Sim


In [32]:
len(gdf_merged)

5572

In [35]:
gdf_merged.to_file('brasil.geojson', driver = 'GeoJSON')